In [ ]:
import os
import glob
import torch
import matplotlib.pyplot as plt

# 1. Define the directory containing your .pth files
checkpoint_dir = "../data/checkpoints-rec/all_inrs_eight_neurons"
search_pattern = os.path.join(checkpoint_dir, "**/*.pth")
pth_files = glob.glob(search_pattern)

all_weights = []

print(f"Found {len(pth_files)} .pth files. Extracting weights...")

# 2. Loop through all files and collect the weights
for file_path in pth_files:
    # load_location='cpu' prevents CUDA out-of-memory issues
    checkpoint = torch.load(file_path, map_location='cpu')
    
    # If your .pth file saves the state_dict directly, use checkpoint.items()
    # If it saves a dict like {'state_dict': ..., 'epoch': 1}, change to checkpoint['state_dict'].items()
    state_dict = checkpoint if 'state_dict' not in checkpoint else checkpoint['state_dict']
    
    for key, val in state_dict.items():
        # Only collect weights (skip biases, running means, or layer_ids if they exist)
        if 'weight' in key and isinstance(val, torch.Tensor):
            # Flatten to 1D and convert to float32 to save memory / ensure consistency
            all_weights.append(val.flatten().to(torch.float32))

# 3. Combine everything into a single 1D tensor
if all_weights:
    combined_weights = torch.cat(all_weights)
    print(f"Successfully collected {combined_weights.numel():,} total weight parameters.")
    
    # 4. Plot the distribution
    plt.figure(figsize=(10, 6))
    
    # Convert to numpy for matplotlib. 100 bins usually gives a smooth curve.
    plt.hist(combined_weights.numpy(), bins=100, color='royalblue', edgecolor='black', alpha=0.7)
    
    plt.title("Distribution of Weights Across Checkpoints", fontsize=14)
    plt.xlabel("Weight Value", fontsize=12)
    plt.ylabel("Frequency", fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Show the plot
    plt.show()
else:
    print("No weights found! Double check your state_dict keys.")

Found 0 .pth files. Extracting weights...
No weights found! Double check your state_dict keys.
